# Exercise 2: Save Deduplicated Results to Apache Iceberg

## Learning Objectives

In this exercise, you will:
- Connect Spark via a **CAI Data Connection** (preferred) or a manual Iceberg REST catalog
- Load either Exercise 1 Parquet **or** the project local CSV
- Write those results into an Iceberg table (different table name per source)
- Find and verify the table in the catalog

## Prerequisites

1. Run **`00_Getting_Started.ipynb`**. For the Parquet path, also run **`01_Basic_Deduplication.ipynb`**.
2. Input options (set `INPUT_SOURCE` in Step 1):

| `INPUT_SOURCE` | File | Iceberg table (default) |
|----------------|------|-------------------------|
| `exercise1_parquet` | `/tmp/.../exercise1_exact.parquet` | `cdp_user_demo.deduped_customers` |
| `local_csv` | `../data/redundant_data.csv` | `cdp_user_demo.raw_customers` |

3. Prefer a **Spark Data Lake / Iceberg** connection in Project Settings → Data Connections (e.g. `go01-obsr-de`). Start the session with **Enable Spark → Spark 3**.
4. Manual REST fallback only if you have a real REST URI + credential (not the `<DATALAKE-HOSTNAME>` placeholder).


## Step 1: Configure Connection and Input Source

Set the CAI Data Connection name (preferred). Choose `INPUT_SOURCE` to load Exercise 1 Parquet or the project local CSV (writes a differently named Iceberg table). Optional REST catalog values are used only when CML Spark is unavailable.


In [ ]:
import os
from pathlib import Path

# --- Preferred: CAI Project Data Connection (Spark Data Lake / Iceberg) ---
CONNECTION_NAME = os.environ.get("CML_CONNECTION_NAME", "go01-obsr-de")

# --- Input source: "exercise1_parquet" (default) or "local_csv" ---
INPUT_SOURCE = os.environ.get("ICEBERG_INPUT_SOURCE", "exercise1_parquet").strip().lower()
# Flip to local CSV without env vars:
# INPUT_SOURCE = "local_csv"

NAMESPACE = os.environ.get("ICEBERG_NAMESPACE", "cdp_user_demo")

# Distinct Iceberg table names per source
TABLE_NAME_DEDUPED = os.environ.get("ICEBERG_TABLE_DEDUPED", "deduped_customers")
TABLE_NAME_RAW = os.environ.get("ICEBERG_TABLE_RAW", "raw_customers")

LOCAL_CSV = Path(os.environ.get("LOCAL_CSV", "../data/redundant_data.csv")).resolve()
LOCAL_PARQUET = Path(
    os.environ.get(
        "EXERCISE1_PARQUET",
        "/tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet",
    )
)

if INPUT_SOURCE == "local_csv":
    INPUT_FORMAT = "csv"
    LOCAL_INPUT = LOCAL_CSV
    TABLE_NAME = TABLE_NAME_RAW
elif INPUT_SOURCE in ("exercise1_parquet", "parquet"):
    INPUT_FORMAT = "parquet"
    LOCAL_INPUT = LOCAL_PARQUET
    TABLE_NAME = TABLE_NAME_DEDUPED
else:
    raise ValueError(
        f"Unknown INPUT_SOURCE={INPUT_SOURCE!r}. Use 'exercise1_parquet' or 'local_csv'."
    )

# Optional override of the resolved table name
TABLE_NAME = os.environ.get("ICEBERG_TABLE", TABLE_NAME)
FULL_TABLE = f"{NAMESPACE}.{TABLE_NAME}"
INPUT_PATH = LOCAL_INPUT.resolve().as_uri() if LOCAL_INPUT.exists() else str(LOCAL_INPUT)

# --- Optional fallback: manual Iceberg REST catalog ---
# Leave unset until you have a real URI. Placeholders break ANY Spark action as defaultCatalog.
CATALOG_NAME = os.environ.get("ICEBERG_CATALOG_NAME", "iceberg")
REST_URI = os.environ.get("ICEBERG_REST_URI", "").strip()
REST_CREDENTIAL = os.environ.get("ICEBERG_REST_CREDENTIAL", "").strip()
REST_URI_CONFIGURED = bool(REST_URI) and "<" not in REST_URI and ">" not in REST_URI

print(f"CML connection: {CONNECTION_NAME}")
print(f"Input source:   {INPUT_SOURCE} ({INPUT_FORMAT})")
print(f"Input path:     {INPUT_PATH}")
print(f"Input exists:   {LOCAL_INPUT.exists()}")
print(f"Target table:   {FULL_TABLE}")
print(f"REST URI set:   {REST_URI_CONFIGURED} ({'yes' if REST_URI_CONFIGURED else 'no — CML Spark preferred; REST write disabled'})")
print(f"REST credential:{'set' if REST_CREDENTIAL else 'not set'}")


## Step 2: Create Spark Session

1. Try `cml.data_v1.get_connection(...).get_spark_session()` (project Data Connection).
2. Else fall back to local Spark for Parquet load only.
3. Manual Iceberg REST catalog is attached **only** when `ICEBERG_REST_URI` is a real URL — never with a `<placeholder>`.


In [ ]:
import os
import pyspark
from pyspark.sql import SparkSession

spark = None
conn = None
SPARK_MODE = None  # "cml_spark" | "local_rest" | "local_parquet"

# 1) Preferred: CAI Data Connection Spark session (Iceberg + cluster Hadoop already wired)
try:
    import cml.data_v1 as cmldata

    conn = cmldata.get_connection(CONNECTION_NAME)
    get_spark = getattr(conn, "get_spark_session", None)
    if callable(get_spark):
        spark = get_spark()
        SPARK_MODE = "cml_spark"
        print(f"✓ Spark from CML connection: {CONNECTION_NAME}")
    else:
        # SQL-only connection — useful for discovery, not for writeTo(iceberg)
        try:
            sample = conn.get_pandas_dataframe("show databases")
            print(f"✓ CML connection '{CONNECTION_NAME}' is SQL-only (pandas). Sample databases:")
            print(sample)
        except Exception as e:
            print(f"⚠ CML connection '{CONNECTION_NAME}' has no get_spark_session: {e}")
        print("→ Falling back to local Spark for Parquet load")
except ImportError:
    print("⚠ cml.data_v1 not available — falling back to local Spark")
except Exception as e:
    print(f"⚠ CML get_connection failed ({e}) — falling back to local Spark")

# 2) Local Spark fallback (do NOT set a broken REST defaultCatalog)
if spark is None:
    for _k in ("HADOOP_CONF_DIR", "HADOOP_HOME", "HADOOP_HDFS_HOME"):
        if _k in os.environ:
            print(f"⚠ Unsetting {_k}={os.environ[_k]} for local Spark startup")
            os.environ.pop(_k)

    _spark_mm = ".".join(pyspark.__version__.split(".")[:2])
    ICEBERG_PACKAGE = os.environ.get(
        "ICEBERG_SPARK_PACKAGE",
        f"org.apache.iceberg:iceberg-spark-runtime-{_spark_mm}_2.12:1.6.1",
    )

    builder = (
        SparkSession.builder
        .master("local[*]")
        .appName("Exercise2_Iceberg")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.hadoop.hadoop.security.authentication", "simple")
        .config("spark.hadoop.hadoop.security.authorization", "false")
        .config("spark.jars.packages", ICEBERG_PACKAGE)
        .config(
            "spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        )
        # Keep built-in catalog as default so Parquet count/show never hits REST
        .config("spark.sql.defaultCatalog", "spark_catalog")
    )

    if REST_URI_CONFIGURED:
        builder = (
            builder
            .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
            .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
            .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", REST_URI)
            .config(f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", NAMESPACE)
        )
        if REST_CREDENTIAL:
            builder = builder.config(
                f"spark.sql.catalog.{CATALOG_NAME}.credential", REST_CREDENTIAL
            )
        SPARK_MODE = "local_rest"
        # Qualified writes use catalog.db.table
        globals()["FULL_TABLE"] = f"{CATALOG_NAME}.{NAMESPACE}.{TABLE_NAME}"
        print(f"✓ Local Spark + Iceberg REST catalog '{CATALOG_NAME}'")
        print(f"  REST URI: {REST_URI}")
    else:
        SPARK_MODE = "local_parquet"
        print("✓ Local Spark (Parquet only — set CML Spark connection or ICEBERG_REST_URI for Iceberg write)")

    spark = builder.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"Mode:          {SPARK_MODE}")
print(f"Write target:  {FULL_TABLE}")


## Step 3: Load Source Data

Loads based on `INPUT_SOURCE` from Step 1:
- `exercise1_parquet` → Exercise 1 deduped Parquet
- `local_csv` → project `../data/redundant_data.csv`


In [ ]:
assert LOCAL_INPUT.exists(), (
    f"Missing input for INPUT_SOURCE={INPUT_SOURCE!r}: {LOCAL_INPUT}\n"
    + (
        "Re-run 01_Basic_Deduplication.ipynb, or set EXERCISE1_PARQUET."
        if INPUT_FORMAT == "parquet"
        else "Ensure use-case-phase-1/data/redundant_data.csv is in the project, or set LOCAL_CSV."
    )
)

if INPUT_FORMAT == "csv":
    df = spark.read.csv(INPUT_PATH, header=True, inferSchema=True)
else:
    df = spark.read.parquet(INPUT_PATH)

print(f"✓ Loaded ({INPUT_FORMAT}): {INPUT_PATH}")
print(f"Will write Iceberg table: {FULL_TABLE}")
print(f"Rows: {df.count():,}")
print(f"Columns: {', '.join(df.columns)}")
df.show(10, truncate=False)
df.printSchema()


## Step 4: Create Namespace and Write Iceberg Table

Requires **CML Spark** (`SPARK_MODE=cml_spark`) or a real `ICEBERG_REST_URI`. Local Parquet-only mode stops here after Step 3.


In [ ]:
assert SPARK_MODE in ("cml_spark", "local_rest"), (
    "Iceberg write needs a Spark Data Lake connection (get_spark_session) "
    "or a real ICEBERG_REST_URI (no <placeholders>). "
    f"Current mode: {SPARK_MODE}"
)

# Database / namespace
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NAMESPACE}")
print(f"✓ Namespace ready: {NAMESPACE}")

(
    df.writeTo(FULL_TABLE)
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print(f"✓ Iceberg table written: {FULL_TABLE}")


## Step 5: Find the Table in the Catalog

List namespaces/tables and confirm the new table is discoverable.


In [ ]:
print("=== Databases / namespaces ===")
spark.sql("SHOW NAMESPACES").show(truncate=False)

print(f"=== Tables in {NAMESPACE} ===")
tables_df = spark.sql(f"SHOW TABLES IN {NAMESPACE}")
tables_df.show(truncate=False)

rows = tables_df.collect()
table_names = []
for r in rows:
    d = r.asDict()
    table_names.append(d.get("tableName") or d.get("name") or d.get("table") or str(r[0]))

found = TABLE_NAME in table_names or any(TABLE_NAME == str(n) for n in table_names)
print(f"Looking for table: {TABLE_NAME}")
print(f"Tables found: {table_names}")
print("✓ Table found in catalog" if found else "✗ Table NOT found — check write step / namespace")


In [ ]:
print("=== DESCRIBE TABLE ===")
spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_TABLE}").show(100, truncate=False)

print("=== Sample query from Iceberg table ===")
iceberg_df = spark.table(FULL_TABLE)
print(f"Rows in Iceberg table: {iceberg_df.count():,}")
iceberg_df.show(10, truncate=False)


## Step 6 (Optional): Query the REST Catalog HTTP API Directly

Only when `ICEBERG_REST_URI` is configured. Skip for CML Spark mode — use Step 5 Spark SQL instead.


In [ ]:
import json
import urllib.request
import urllib.error
import base64

if not REST_URI_CONFIGURED:
    print("Skipped — ICEBERG_REST_URI not set. Use Step 5 Spark SQL for catalog discovery.")
else:
    def rest_get(path: str):
        """GET a path under the Iceberg REST catalog URI."""
        url = REST_URI.rstrip("/") + path
        req = urllib.request.Request(url, method="GET")
        req.add_header("Accept", "application/json")

        token = os.environ.get("ICEBERG_REST_TOKEN", "")
        if token:
            req.add_header("Authorization", f"Bearer {token}")
        elif REST_CREDENTIAL:
            encoded = base64.b64encode(REST_CREDENTIAL.encode("utf-8")).decode("ascii")
            req.add_header("Authorization", f"Basic {encoded}")

        with urllib.request.urlopen(req, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        ns_path = NAMESPACE.replace(".", "%1F")
        payload = rest_get(f"/v1/namespaces/{ns_path}/tables")
        identifiers = payload.get("identifiers", [])
        print("REST API tables in namespace:")
        print(json.dumps(identifiers, indent=2))

        matched = [
            i for i in identifiers
            if i.get("name") == TABLE_NAME or TABLE_NAME in str(i)
        ]
        if matched:
            print(f"\n✓ Found via REST API: {matched}")
            meta = rest_get(f"/v1/namespaces/{ns_path}/tables/{TABLE_NAME}")
            print("\nTable metadata keys:", list(meta.keys()))
        else:
            print(f"\n✗ Table '{TABLE_NAME}' not returned by REST list endpoint")
    except urllib.error.HTTPError as e:
        print(f"REST API HTTP error: {e.code} {e.reason}")
        print("Spark SQL discovery in Step 5 may still succeed.")
    except Exception as e:
        print(f"REST API call skipped/failed: {e}")


## Summary

| Item | Value |
|------|-------|
| Input option A | Exercise 1 Parquet → `cdp_user_demo.deduped_customers` |
| Input option B | Local CSV `../data/redundant_data.csv` → `cdp_user_demo.raw_customers` |
| Preferred Spark | CML Data Connection `go01-obsr-de` via `get_spark_session()` |
| REST fallback | only when `ICEBERG_REST_URI` is a real URL |

### Key Takeaways

- Do **not** set Iceberg as `defaultCatalog` with a placeholder REST URI — even `df.count()` on Parquet will fail
- Prefer CAI Data Connections for Iceberg + cluster Hadoop wiring
- Use a **different Iceberg table name** per source so raw and deduped data do not overwrite each other
- `SHOW TABLES` / `DESCRIBE TABLE` confirm registration after write

## Cleanup


In [ ]:
spark.stop()
if conn is not None and hasattr(conn, "close"):
    try:
        conn.close()
    except Exception:
        pass
print("✓ Spark session stopped")
